In [2]:
!pip install faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 34.4 MB/s eta 0:00:00


In [3]:
import pandas as pd
import random
import uuid
import zipfile
import os
from faker import Faker

fake = Faker('en_US')

In [4]:
# ==========================================
# 1. DYNAMIC PARAMETERS (SIMULATION CONTROL)
# ==========================================
TOTAL_EMPLOYEES = 5000  # Total active headcount
OPEN_VACANCIES = 500    # Total budget slots unassigned (Open Positions)

DEPARTMENTS = [
    'Marketing', 'Sales', 'Engineering', 'IT', 'Risk & Compliance',
    'Finance', 'R&D', 'People', 'Operations', 'Logistics'
]

POP_CULTURE_NAMES = [
    "Thomas Anderson", "Agent Smith", "Morpheus", "Trinity",
    "Billy Butcher", "Hughie Campbell", "Annie January", "Homelander",
    "Luke Skywalker", "Leia Organa", "Obi-one Kenobi", "Darth Vader",
    "Uzumaki Naruto", "Uchiha Sasuke", "Uchiha Itachi", "Haruno Sakura",
    "Michael Scott", "Dwight Schrute", "Jim Halpert", "Pam Beesly",
    "Jake Peralta", "Raymond Holt", "Terry Jeffords", "Rosa Diaz"
]

SKILLS_POOL = {
    'IT': ['Python', 'Cloud Infrastructure (AWS/GCP)', 'Cybersecurity', 'System Architecture', 'API Development'],
    'Engineering': ['EV Powertrain Engineering', 'CAD Component Design', 'Vehicle Dynamics Simulation', 'Robotics'],
    'Operations': ['Lean Manufacturing & Kaizen', 'Six Sigma', 'Supply Chain Optimization', 'Quality Assurance'],
    'Sales': ['B2B Negotiation', 'CRM Management', 'Sales Forecasting', 'Lead Generation'],
    'Finance': ['Financial Modeling', 'Risk Analysis', 'Corporate Taxation', 'Asset Management'],
    'Marketing': ['Growth Hacking', 'SEO/SEM', 'Brand Strategy', 'Content Marketing'],
    'People': ['Talent Acquisition', 'Organizational Development', 'Conflict Resolution', 'HR Analytics'],
    'R&D': ['Rapid Prototyping', 'Innovation Strategy', 'Material Science', 'Data Modeling'],
    'Risk & Compliance': ['Regulatory Compliance', 'Internal Auditing', 'Fraud Detection', 'Data Privacy'],
    'Logistics': ['Fleet Management', 'Inventory Control', 'Warehouse Operations', 'Route Optimization'],
    'Hybrid': ['Data Analysis (SQL/BI)', 'Agile/Scrum', 'Project Management', 'Business Strategy', 'Design Thinking'],
    'Soft_Skills': ['Emotional Intelligence', 'Critical Thinking', 'Complex Problem Solving', 'Adaptability', 'Public Speaking']
}


In [5]:
# ==========================================
# 2. GENERATE SKILLS NODES (EXACTLY 51 NODES)
# ==========================================
print("Generating baseline competency catalog...")
skills_data = []
skill_id_map = {}
for category, skills in SKILLS_POOL.items():
    for skill in skills:
        s_id = f"SKILL_{uuid.uuid4().hex[:8].upper()}"
        skills_data.append({"id": s_id, "skill_name": skill, "category": category})
        skill_id_map[skill] = s_id
df_skills_nodes = pd.DataFrame(skills_data)

# Skill transitions metadata matrix
equivalent_data = [
    {"skill_id_1": skill_id_map['Python'], "skill_id_2": skill_id_map['Data Analysis (SQL/BI)'], "similarity": 0.8, "reskill_time_weeks": 2},
    {"skill_id_1": skill_id_map['Financial Modeling'], "skill_id_2": skill_id_map['Data Analysis (SQL/BI)'], "similarity": 0.7, "reskill_time_weeks": 3},
    {"skill_id_1": skill_id_map['Project Management'], "skill_id_2": skill_id_map['Agile/Scrum'], "similarity": 0.9, "reskill_time_weeks": 1}
]
df_equivalent = pd.DataFrame(equivalent_data)


Generating baseline competency catalog...


In [6]:
# ==========================================
# 3. GENERATE ROLE STANDARD PROFILES & SKILL REQUIREMENTS
# ==========================================
print("Synthesizing baseline Role catalog and structural skill requirements...")
roles_data = []
requires_skill_data = []

# Generate standardized roles per level and department
role_catalog = {}

# CEO Profile
roles_data.append({"id": "ROLE_CEO", "title": "Chief Executive Officer", "level": "C-Level"})
for s in SKILLS_POOL['Soft_Skills'] + SKILLS_POOL['Hybrid']:
    requires_skill_data.append({"role_id": "ROLE_CEO", "skill_id": skill_id_map[s]})

for dept in DEPARTMENTS:
    # C-Level Profile
    c_role_id = f"ROLE_{dept.upper()}_C_LEVEL"
    roles_data.append({"id": c_role_id, "title": f"Chief {dept} Officer", "level": "C-Level"})
    role_catalog[(dept, 'C-Level')] = c_role_id

    # Director Profile
    d_role_id = f"ROLE_{dept.upper()}_DIRECTOR"
    roles_data.append({"id": d_role_id, "title": f"Director of {dept}", "level": "Director"})
    role_catalog[(dept, 'Director')] = d_role_id

    # Manager Profile
    m_role_id = f"ROLE_{dept.upper()}_MANAGER"
    roles_data.append({"id": m_role_id, "title": f"{dept} Manager", "level": "Manager"})
    role_catalog[(dept, 'Manager')] = m_role_id

    # Operational Profiles (Specialist, Senior, Junior)
    for lvl in ['Specialist', 'Senior', 'Junior']:
        o_role_id = f"ROLE_{dept.upper()}_{lvl.upper()}"
        title = f"{dept} Specialist" if lvl == 'Specialist' else f"{lvl} Analyst - {dept}"
        roles_data.append({"id": o_role_id, "title": title, "level": lvl})
        role_catalog[(dept, lvl)] = o_role_id

        # Connect Roles to specific domain skills
        for s in SKILLS_POOL[dept] + random.sample(SKILLS_POOL['Hybrid'], 2):
            requires_skill_data.append({"role_id": o_role_id, "skill_id": skill_id_map[s]})

df_roles_nodes = pd.DataFrame(roles_data)
df_requires_skill = pd.DataFrame(requires_skill_data)

Synthesizing baseline Role catalog and structural skill requirements...


In [7]:
# ==========================================
# 4. TOP-DOWN DETERMINISTIC STRUCTURAL TREE
# ==========================================
print("Building rigid organizational tree matrix...")
employees_raw = []
reports_to_data = []
holds_role_data = []

# Instantiate CEO
ceo_email = "thomas.anderson.E0001@neosmith.com"
employees_raw.append({
    "id": ceo_email, "name": "Thomas Anderson",
    "department": "Executive", "influence_score": 0.95
})
holds_role_data.append({"employee_id": ceo_email, "role_id": "ROLE_CEO"})

pop_culture_copy = POP_CULTURE_NAMES.copy()
if "Thomas Anderson" in pop_culture_copy:
    pop_culture_copy.remove("Thomas Anderson")

emp_counter = 2
managers_pool = {dept: [] for dept in DEPARTMENTS}

# Build out Management Hierarchy Layer
for dept in DEPARTMENTS:
    # 1 C-Level per Dept
    c_name = pop_culture_copy.pop(0) if pop_culture_copy else fake.name()
    c_email = f"{c_name.lower().replace(' ', '.')}.E{emp_counter:04d}@neosmith.com"
    employees_raw.append({"id": c_email, "name": c_name, "department": dept, "influence_score": round(random.uniform(0.65, 0.85), 2)})
    holds_role_data.append({"employee_id": c_email, "role_id": role_catalog[(dept, 'C-Level')]})
    reports_to_data.append({"subordinate_id": c_email, "manager_id": ceo_email})
    emp_counter += 1

    # 2 Directors per C-Level
    for _ in range(2):
        d_name = pop_culture_copy.pop(0) if pop_culture_copy else fake.name()
        d_email = f"{d_name.lower().replace(' ', '.')}.E{emp_counter:04d}@neosmith.com"
        employees_raw.append({"id": d_email, "name": d_name, "department": dept, "influence_score": round(random.uniform(0.50, 0.75), 2)})
        holds_role_data.append({"employee_id": d_email, "role_id": role_catalog[(dept, 'Director')]})
        reports_to_data.append({"subordinate_id": d_email, "manager_id": c_email})
        emp_counter += 1

        # 3 Managers per Director
        for _ in range(3):
            m_name = pop_culture_copy.pop(0) if pop_culture_copy else fake.name()
            m_email = f"{m_name.lower().replace(' ', '.')}.E{emp_counter:04d}@neosmith.com"
            employees_raw.append({"id": m_email, "name": m_name, "department": dept, "influence_score": round(random.uniform(0.40, 0.65), 2)})
            holds_role_data.append({"employee_id": m_email, "role_id": role_catalog[(dept, 'Manager')]})
            reports_to_data.append({"subordinate_id": m_email, "manager_id": d_email})
            managers_pool[dept].append(m_email)
            emp_counter += 1

Building rigid organizational tree matrix...


In [8]:
# ==========================================
# 5. ALLOCATE OPERATIONAL FORCE & VACANCIES
# ==========================================
print("Distributing operational layers and managing open headcount budgets...")
open_positions_raw = []
for_role_data = []

total_operational_slots = (TOTAL_EMPLOYEES - emp_counter + 1) + OPEN_VACANCIES
operational_mix = ['Specialist'] * int(total_operational_slots * 0.25) + \
                  ['Senior'] * int(total_operational_slots * 0.35) + \
                  ['Junior'] * (total_operational_slots - int(total_operational_slots * 0.25) - int(total_operational_slots * 0.35))
random.shuffle(operational_mix)

# Mark specific slots as open vacancies
vacancy_flags = [False] * (TOTAL_EMPLOYEES - emp_counter + 1) + [True] * OPEN_VACANCIES
random.shuffle(vacancy_flags)

vac_counter = 1
for is_vacancy in vacancy_flags:
    dept = DEPARTMENTS[emp_counter % 10]
    assigned_manager = random.choice(managers_pool[dept])
    level = operational_mix.pop()
    target_role_id = role_catalog[(dept, level)]

    if is_vacancy:
        vac_id = f"VAC_{vac_counter:04d}"
        open_positions_raw.append({"id": vac_id, "department": dept, "priority": random.choice(['High', 'Medium', 'Low'])})
        for_role_data.append({"position_id": vac_id, "role_id": target_role_id})
        reports_to_data.append({"subordinate_id": vac_id, "manager_id": assigned_manager})  # Vacancy reports to Manager
        vac_counter += 1
    else:
        e_name = fake.name()
        e_email = f"{e_name.lower().replace(' ', '.')}.E{emp_counter:04d}@neosmith.com"

        # Enforce 5% hidden influencers ONA metric rule
        inf_score = round(random.uniform(0.75, 0.90), 2) if random.random() < 0.05 else round(random.gauss(0.4, 0.1), 2)
        inf_score = max(0.01, min(0.99, inf_score))

        employees_raw.append({"id": e_email, "name": e_name, "department": dept, "influence_score": inf_score})
        holds_role_data.append({"employee_id": e_email, "role_id": target_role_id})
        reports_to_data.append({"subordinate_id": e_email, "manager_id": assigned_manager})
        emp_counter += 1

df_employees_nodes = pd.DataFrame(employees_raw)
df_open_positions_nodes = pd.DataFrame(open_positions_raw)
df_holds_role = pd.DataFrame(holds_role_data)
df_for_role = pd.DataFrame(for_role_data)
df_reports_to = pd.DataFrame(reports_to_data)

Distributing operational layers and managing open headcount budgets...


In [9]:
# ==========================================
# 6. CONNECT SKILLS & COLLABORATION
# ==========================================
print("Populating skill proficiency assignments and informal networks...")
has_skill_data = []
for emp in employees_raw:
    dept_skills = SKILLS_POOL.get(emp['department'], [])
    all_possible = dept_skills + SKILLS_POOL['Hybrid'] + SKILLS_POOL['Soft_Skills']
    for s_name in random.sample(all_possible, min(random.randint(3, 5), len(all_possible))):
        has_skill_data.append({
            "employee_id": emp['id'], "skill_id": skill_id_map[s_name], "proficiency": random.randint(2, 5)
        })
df_has_skill = pd.DataFrame(has_skill_data)

collaborates_data = []
for _ in range(15000):
    e1 = random.choice(employees_raw)
    e2 = random.choice(employees_raw)
    if e1['id'] != e2['id']:
        collaborates_data.append({
            "emp_id_1": e1['id'], "emp_id_2": e2['id'], "weight": round(random.uniform(0.25, 0.95), 2)
        })
df_collaborates = pd.DataFrame(collaborates_data)

Populating skill proficiency assignments and informal networks...


In [10]:
# ==========================================
# 7. ARCHIVE COMPRESSION PACKAGING
# ==========================================
csv_files = {
    "import_nodes_employees.csv": df_employees_nodes,
    "import_nodes_roles.csv": df_roles_nodes,
    "import_nodes_open_positions.csv": df_open_positions_nodes,
    "import_nodes_skills.csv": df_skills_nodes,
    "import_edges_holds_role.csv": df_holds_role,
    "import_edges_for_role.csv": df_for_role,
    "import_edges_reports_to.csv": df_reports_to,
    "import_edges_has_skill.csv": df_has_skill,
    "import_edges_requires_skill.csv": df_requires_skill,
    "import_edges_equivalent_to.csv": df_equivalent,
    "import_edges_collaborates.csv": df_collaborates
}

print("Exporting localized operational CSV matrices...")
for filename, df in csv_files.items():
    df.to_csv(filename, index=False)

zip_filename = "neosmith_target_graph.zip"
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for filename in csv_files.keys():
        zipf.write(filename)
        os.remove(filename)

print(f"\nExecution Complete! Packet '{zip_filename}' generated smoothly.")
print(f"Active Employees mapped: {len(df_employees_nodes)} | Open Slots created: {len(df_open_positions_nodes)}")

Exporting localized operational CSV matrices...

Execution Complete! Packet 'neosmith_target_graph.zip' generated smoothly.
Active Employees mapped: 5000 | Open Slots created: 500
